In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

<h2 style="color:Red"> Pre-Processing


<h4 style="color:#0B6C03"> 1. Converting to Lower Case


In [8]:
df["review"] = df["review"].str.lower()

<h4 style="color:#0B6C03"> 2. Remove URL(HTTP/HTTPS) , Punctuations, HTML


In [9]:
# Regular Expression Library
import re

<h5 style="color:#b2b131"> i. Remove URL(HTTP/HTTPS)


In [10]:
def remove_urls(text):
    text = re.sub(pattern=r"http\S+", repl="", string=text)
    return text


df["review"] = df["review"].apply(remove_urls)

<h5 style="color:#b2b131"> ii. Remove Punctuations


In [11]:
# not in A-Z, a-z, 0-9, all kind of white spaces
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text


df["review"] = df["review"].apply(remove_punctuations)

<h5 style="color:#b2b131"> iii. Remove HTML tags


In [12]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text


df["review"] = df["review"].apply(remove_html)

<h4 style="color:#0B6C03"> 3. Remove stopwords (i.e. general english words)


NLTK => Natural Language Tool Kit


<h5> Download</h5>


In [13]:
import nltk
import sys

download_path = sys.prefix + r"\nltk_data"
print(download_path)

nltk.download("punkt", download_dir=download_path)
nltk.download("punkt_tab", download_dir=download_path)
nltk.download("stopwords", download_dir=download_path)

d:\tools\Anaconda\envs\deep_learning\nltk_data


[nltk_data] Downloading package punkt to
[nltk_data]     d:\tools\Anaconda\envs\deep_learning\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     d:\tools\Anaconda\envs\deep_learning\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     d:\tools\Anaconda\envs\deep_learning\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

<h5> Verify


In [14]:
nltk.data.path.append(sys.prefix + r"\nltk_data")

print(nltk.data.find("tokenizers/punkt"))
print(nltk.data.find("tokenizers/punkt_tab"))
print(nltk.data.find("corpora/stopwords"))

d:\tools\Anaconda\envs\deep_learning\nltk_data\tokenizers\punkt
d:\tools\Anaconda\envs\deep_learning\nltk_data\tokenizers\punkt_tab
d:\tools\Anaconda\envs\deep_learning\nltk_data\corpora\stopwords


In [15]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [16]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text


df["review"] = df["review"].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


<h4 style="color:#0B6C03"> 4. Stemming & Lemmatization


    Stemming =>
    eg.
        running -> run
        played -> play

    use PorterStemming


In [18]:
from nltk.stem import PorterStemmer


def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)


df["review"] = df["review"].apply(stemming)

In [19]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


<h4 style="color:#0B6C03"> 5. Encode Sentiment


In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [21]:
y = df["sentiment"]

<h4 style="color:#0B6C03"> 6. Vectorization (TF-IDF)


    Convert Text data to Numbers.
    is known as Vectorization


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

# max_features =>
# Build a vocabulary that only consider the top max_features
# ordered by term frequency across the corpus.
# Otherwise, all features are used.

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

<h2 style="color:Red"> Dataset & Data Loaders


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
X_train.shape

(39665, 5000)

In [25]:
X_test.shape

(9917, 5000)

In [26]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [27]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [28]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(), torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(), torch.from_numpy(y_test.values).float()
)

C:\Users\PD Patel\AppData\Local\Temp\ipykernel_12112\3954900532.py:2: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(X_train).float(), torch.from_numpy(y_train.values).float()


In [29]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

<h2 style="color:Red"> Build RNN

In [30]:
import torch.nn as nn
import torch.optim as optim

In [31]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully Connected Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps ===> out => (batch, seq_len, hidden_size)
        # 2nd value = final hidden state of last timestep ===> _

        out = self.fc(out[:, -1, :])

        return out

In [32]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

<h2 style="color:Red"> Training RNN

In [33]:
# Unsqueeze Squeeze

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        # unsqueeze => add singleton direction
        # to add one direction and make 2D matrix to 3D
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        # size => (batch_size, 1)

        # squeeze => reduce by one direction
        # i.e. 3D to 2D
        outputs = torch.sigmoid(outputs.squeeze(1))  # size => (batch_size)

        loss = criterion(outputs, yb)

        # Back propagation
        loss.backward()

        # Weight update
        optimizer.step()

    print(f"epoch {epoch+1}/{epochs} ==> loss = {loss.item()}")

epoch 1/10 ==> loss = 0.2161041498184204
epoch 2/10 ==> loss = 0.14322113990783691
epoch 3/10 ==> loss = 0.23646825551986694
epoch 4/10 ==> loss = 0.2424391657114029
epoch 5/10 ==> loss = 0.3313434422016144
epoch 6/10 ==> loss = 0.19910050928592682
epoch 7/10 ==> loss = 0.21969999372959137
epoch 8/10 ==> loss = 0.38232889771461487
epoch 9/10 ==> loss = 0.28097665309906006
epoch 10/10 ==> loss = 0.23121440410614014


<h2 style="color:Red"> Evaluate RNN

In [34]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy= {correct_vals/tot_vals*100}")

accuracy= 85.5803166280125
